In [ ]:
pip install chromadb==0.4.22

In [1]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline


pipe = pipeline("text2text-generation", model="google/flan-t5-base")
llm = HuggingFacePipeline(pipeline=pipe)

/Users/Sandras/Desktop/Python/FLEXISAF/Final_project/langchain_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## create a prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful school assistant"),
    ("human", "{user_input}")
])

def messages(user_input):
    return prompt.format_messages(user_input=user_input)



In [3]:
## time to prepare my file.
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="Final_files/school_file.json",
    jq_schema=".[] | .content"
)
documents = loader.load()

## now we chunk
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200,add_start_index=True)
allsplit=text_splitter.split_documents(documents)


In [15]:
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

embed = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


vectorstore = Chroma.from_documents(
    documents=allsplit,
    embedding=embed,
    persist_directory="new_db"
)

vectorstore.persist()

results = vectorstore.similarity_search("What is the tuition fee?")
print(results[0])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


page_content='Tuition fees vary by program. Undergraduate programs range from ₦500,000 to ₦1,200,000 per academic session. Payments can be made via bank transfer, online payment portal, or at designated bank branches. Students are allowed to pay in two installments. Scholarships and financial aid are available for outstanding students based on merit and need.' metadata={'seq_num': 4, 'source': '/Users/Sandras/Desktop/Python/FLEXISAF/Final_project/Final_files/school_file.json', 'start_index': 0}
